# Notebook 27 — both open judges on all 300. EXPLORATORY DIAGNOSTIC.

> **This is not a scorer and nothing here may be reported as corrected
> accuracy.** Both judges already failed a safety check, and the two failures
> are different in kind:
>
> | judge | gate | outcome |
> |---|---|---|
> | `jnanliu/LiveMath-Judge` | verdict | **FAILED** — accepted item 273's non-answer |
> | `KbsdJames/Omni-Judge` | verdict | passed |
> | `KbsdJames/Omni-Judge` | rationale | **FAILED** — right verdict, comparison never performed |
>
> Omni-Judge's justification on item 273 read *"The reference answer is 3/4"*.
> We passed `2/4` as the reference and the model answered a bare `}`. It had
> reconstructed a reference from the problem text and graded against that.

**So why run them at all?** Because both findings are currently single items.
An all-300 pass converts two anecdotes into **rates**: how often the judges
disagree with each other, how often each disagrees with the frozen rules, and
how often Omni's justification cites a number appearing in **none** of its
three inputs. That last is the item-273 signature made countable, and it is
the thing this notebook exists for.

**Qwen is not re-run.** Both judges are fed the stored majority answer from
`scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv`, computed
**once** and handed to both, so the comparison is like-for-like by
construction rather than by hope.

### The trap this notebook is built around

Agreement between two unreliable scorers measures neither of them. Worse,
`strict_v1` calls 47% of items correct, so **a judge answering "yes" to
everything agrees with it on 47% for free** — and notebook 25 already found
LiveMath-Judge sitting exactly on that trivial baseline against human labels.
Every agreement figure below therefore ships with `always_yes_agreement`, and
judge-vs-judge agreement with its chance rate and Cohen's kappa. **A raw
agreement number quoted alone from this file is a misuse of it.**

### Prompts

LiveMath-Judge runs the **fidelity-amended** prompt (the one its gate was
decided on). Omni-Judge runs its **native** prompt, because that prompt is
baked into `tokenizer.get_context()` and there is no supported way to amend
it. Notebook 25 found the fidelity clause changed 2 of 40 items and both
toward leniency, so this is not a live variable.

### Cost and shape

~600 generations. The two models are loaded **sequentially** and the first is
freed before the second loads, so this fits a 16 GB GPU rather than needing an
A100 to hold 3B + 8B at once. Both loops checkpoint to Drive every 25 items
and resume, because a disconnect at item 280 must not cost the run.

`strict_v1`, `strict_v2` and the audit CSVs are read only. **No scorer rule is
modified.**

In [1]:
# Auth + code access. GPU: LiveMath-Judge is 3B, Omni-Judge is 8B. They are
# loaded SEQUENTIALLY (see below), so ~16 GB is enough for either alone.
import json
import os
import sys

from google.colab import drive
from huggingface_hub import login

drive.mount("/content/drive")
PROJECT_DIR = "/content/drive/MyDrive/uncertainty-math-vlm"
RESULTS_DIR = f"{PROJECT_DIR}/results"
OUT_DIR = f"{PROJECT_DIR}/audit"
CKPT_DIR = f"{PROJECT_DIR}/checkpoints"
os.makedirs(OUT_DIR, exist_ok=True)
os.makedirs(CKPT_DIR, exist_ok=True)

with open(f"{PROJECT_DIR}/.tokens.json") as f:
    HF_TOKEN = json.load(f)["HF_TOKEN"]
login(token=HF_TOKEN)

REPO_URL = "https://github.com/sepehrmaleki369/uncertainty-math-vlm.git"
!rm -rf repo
!git clone -q {REPO_URL} repo
%pip install -q -e repo/
%pip install -q "antlr4-python3-runtime==4.11"

sys.path.insert(0, os.path.abspath("repo"))
for _name in [m for m in sys.modules if m == "pilot" or m.startswith("pilot.")]:
    del sys.modules[_name]
import importlib
importlib.invalidate_caches()

import pilot.audit_diagnostics
import pilot.canonicalize
import pilot.judge
import pilot.rescore
import pilot.strict_v2

print(f"pilot imported from: {os.path.dirname(pilot.judge.__file__)}")
assert hasattr(pilot.judge, "open_judge_frame"), (
    "the repo clone predates notebook 27's library code -- push pilot/judge.py "
    "and re-run this cell. A green local dry run does NOT cover this: the dry "
    "run uses the working tree, Colab uses the remote.")
assert pilot.canonicalize.latex_parser_available(), (
    "SymPy's LaTeX parser is NOT working -- the strict_v1/strict_v2 columns "
    "below would be computed against degraded labels and every agreement "
    "figure would be measured against the wrong target.")
print("SymPy LaTeX parser OK")

import torch
assert torch.cuda.is_available(), "no GPU: enable a GPU runtime"
print(f"GPU: {torch.cuda.get_device_name(0)}")
print(f"gate items re-derived for free: {pilot.judge.GATE_ITEMS}")

Mounted at /content/drive
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 144.2/144.2 kB 11.7 MB/s eta 0:00:00
  Building editable for pilot (pyproject.toml) ... done
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
omegaconf 2.3.1 requires antlr4-python3-runtime==4.9.*, but you have antlr4-python3-runtime 4.11.0 which is incompatible.
pilot imported from: /content/repo/pilot
SymPy LaTeX parser OK
GPU: NVIDIA A100-SXM4-40GB
gate items re-derived for free: (55, 273)


In [2]:
# The shared inputs. Everything both judges see is built ONCE here.
import ast

import pandas as pd
from tqdm.auto import tqdm

RUN_CSV = "scaleup_n300_bal50_qwen25-vl-3b-instruct_20260802T163202Z.csv"
run = pd.read_csv(f"{RESULTS_DIR}/{RUN_CSV}")
assert len(run) == 300, f"expected 300 rows, got {len(run)}"
print(f"run: {len(run)} rows, model={run['model_id'].unique().tolist()}")

# THE MAJORITY ANSWER, COMPUTED ONCE. Both judges are handed this identical
# string. Letting each judge recompute it would be one refactor away from
# feeding them different inputs, which would make every comparison in this
# notebook meaningless while still producing a plausible-looking table.
answers = pd.Series(
    {i: pilot.judge.majority_answer(
        ast.literal_eval(run.loc[i, "all_transcription_samples_raw"]),
        run.loc[i, "pert_a"])
     for i in tqdm(run.index, desc="majority answers")})
print(f"answers built for {len(answers)} items; "
      f"{int((answers.str.len() == 0).sum())} are empty")

# The frozen rules, RECOMPUTED for comparison only. Neither is modified.
v1 = pilot.rescore.rescore_run(run, "strict_v1")["transcription_correct"].astype(bool)
v2s = pilot.strict_v2.rescore_v2(run, progress=True)
v2 = v2s["correct_strict_v2_display_primary"].astype(bool)
answer_types = v2s.apply(pilot.judge.answer_type, axis=1)
print(f"strict_v1 correct: {int(v1.sum())}/300  |  "
      f"strict_v2 correct: {int(v2.sum())}/300")

# Determinate human labels, deduplicated by the audit sets' own precedence.
audits = pilot.audit_diagnostics.load_audit_sets("repo/reference/audit")
dedup = {}
for name in pilot.audit_diagnostics.SET_PRECEDENCE:
    for i, r in audits[name].iterrows():
        dedup.setdefault(i, r["truth"])
human = pd.Series(dedup)
human = human[human != "indeterminate"]
print(f"determinate human labels: {len(human)} "
      f"({int((human == 'correct').sum())} correct, "
      f"{int((human == 'wrong').sum())} wrong)")
print("\nNOTE: those labels come from TARGETED audit sets, two of which are\n"
      "one-directional by construction. They are context for reading a\n"
      "transcript, never a population against which to score a judge.")

run: 300 rows, model=['Qwen/Qwen2.5-VL-3B-Instruct']


majority answers:   0%|          | 0/300 [00:00<?, ?it/s]

answers built for 300 items; 7 are empty


strict_v2_display_primary:   0%|          | 0/300 [00:00<?, ?it/s]

strict_v1 correct: 141/300  |  strict_v2 correct: 138/300
determinate human labels: 148 (128 correct, 20 wrong)

NOTE: those labels come from TARGETED audit sets, two of which are
one-directional by construction. They are context for reading a
transcript, never a population against which to score a judge.


In [3]:
# Checkpointing. 600 generations is long enough that a disconnect at item
# 280 must not cost the run, and long enough that a stale checkpoint from a
# DIFFERENT model would silently corrupt the comparison -- so the file records
# which model and prompt produced it and refuses to resume across a mismatch.
def ckpt_path(tag):
    return f"{CKPT_DIR}/nb27_{tag}_all300.json"


def load_ckpt(tag, model_id, prompt):
    path = ckpt_path(tag)
    if not os.path.exists(path):
        return {}
    with open(path) as fh:
        blob = json.load(fh)
    if blob.get("model_id") != model_id or blob.get("prompt") != prompt:
        raise RuntimeError(
            f"checkpoint at {path} was written by model={blob.get('model_id')} "
            f"prompt={blob.get('prompt')}, but this session is {model_id}/"
            f"{prompt}. Resuming would mix two judges into one column. Delete "
            "the file deliberately if that is what you want.")
    print(f"resuming {tag} from {len(blob['entries'])} completed items")
    return {int(k): v for k, v in blob["entries"].items()}


def save_ckpt(tag, model_id, prompt, entries):
    with open(ckpt_path(tag), "w") as fh:
        json.dump({"model_id": model_id, "prompt": prompt,
                   "entries": {str(k): v for k, v in entries.items()}}, fh)


def run_judge(tag, model_id, prompt, judge_fn, every=25):
    """Judge all 300 with `judge_fn(item) -> (verdict, raw)`, checkpointed."""
    entries = load_ckpt(tag, model_id, prompt)
    todo = [i for i in run.index if int(i) not in entries]
    for i in tqdm(todo, desc=tag):
        verdict, raw = judge_fn(int(i))
        entries[int(i)] = {"verdict": verdict, "raw_output": raw}
        if len(entries) % every == 0:
            save_ckpt(tag, model_id, prompt, entries)
    save_ckpt(tag, model_id, prompt, entries)
    out = pd.DataFrame.from_dict(entries, orient="index")
    out.index = out.index.astype(int)
    return out.sort_index()


def free_gpu():
    """Reclaim VRAM so the next model fits. Without this the 3B and the 8B are
    resident together and a 16 GB runtime OOMs on the second load.

    Takes NO arguments on purpose. Passing the model in would bind it to a
    parameter and `del` would drop only that local name -- the caller's global
    would still hold the model while `gc.collect()` ran, so this would print a
    reassuring message and free nothing. The caller must `del` its own names
    FIRST and then call this.
    """
    import gc
    gc.collect()
    torch.cuda.empty_cache()
    print(f"GPU reclaimed; reserved now "
          f"{torch.cuda.memory_reserved() / 1e9:.2f} GB")


print("checkpoint helpers ready")

checkpoint helpers ready


In [4]:
# --- JUDGE 1: LiveMath-Judge (3B), fidelity prompt -------------------------
from transformers import AutoModelForCausalLM, AutoTokenizer

LM_ID = pilot.judge.MODEL_ID
lm_tok = AutoTokenizer.from_pretrained(LM_ID, token=HF_TOKEN)
lm_model = AutoModelForCausalLM.from_pretrained(
    LM_ID, torch_dtype=torch.bfloat16, device_map="auto", token=HF_TOKEN).eval()
_lm_pad = lm_tok.pad_token_id or lm_tok.eos_token_id


def livemath_judge(item):
    """Uses the SHARED `answers[item]`, not a per-judge recompute.

    Two corrections to the published snippet are load-bearing and neither
    fails loudly: `return_dict=True` (the card returns a TENSOR then
    subscripts it) and an explicit max_new_tokens (its default of 20 truncates
    before the boxed verdict, so every item would parse as a failure that
    looks like judge breakage).
    """
    prompt = pilot.judge.build_prompt(
        run.loc[item, "orig_q"], run.loc[item, "pert_a"], answers[item],
        fidelity=True)
    inputs = lm_tok.apply_chat_template(
        [{"role": "user", "content": prompt}], return_tensors="pt",
        return_dict=True, add_generation_prompt=True).to(lm_model.device)
    with torch.no_grad():
        out = lm_model.generate(**inputs,
                                max_new_tokens=pilot.judge.MAX_NEW_TOKENS,
                                do_sample=False, pad_token_id=_lm_pad)
    # Only the NEW tokens, so the prompt's own literal \boxed{yes} can never
    # be read as the verdict.
    text = lm_tok.decode(out[0][inputs["input_ids"].shape[1]:],
                         skip_special_tokens=True)
    return pilot.judge.parse_verdict(text), text


livemath = run_judge("livemath", LM_ID, "fidelity", livemath_judge)
print(f"\nLiveMath-Judge: {len(livemath)} items")
print(livemath["verdict"].value_counts().to_string())
print("\ngate items re-derived (expected 55=incorrect, 273=correct):")
for i in pilot.judge.GATE_ITEMS:
    print(f"  item {i}: {livemath.loc[i, 'verdict']}")

# `del` FIRST, then reclaim -- see free_gpu's docstring for why the order is
# not cosmetic. `livemath_judge` still references these globals, but it is
# never called again: the resume path finds all 300 in the checkpoint.
del lm_model, lm_tok
free_gpu()

config.json:   0%|          | 0.00/775 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/7.45k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/2.78M [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/1.67M [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 11.4MB            

...

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/434 [00:00<?, ?it/s]

livemath:   0%|          | 0/300 [00:00<?, ?it/s]


LiveMath-Judge: 300 items
verdict
correct      261
incorrect     39

gate items re-derived (expected 55=incorrect, 273=correct):
  item 55: incorrect
  item 273: correct
GPU reclaimed; reserved now 6.17 GB


In [5]:
# --- JUDGE 2: Omni-Judge (8B), native prompt -------------------------------
# The prompt is baked into tokenizer.get_context(); there is no supported way
# to add a fidelity clause, so this runs native and that is recorded as a
# difference between the two judges rather than papered over.
from huggingface_hub import hf_hub_download
from transformers import PreTrainedTokenizerFast

OM_ID = pilot.judge.OMNI_MODEL_ID
om_tok = AutoTokenizer.from_pretrained(OM_ID, trust_remote_code=True,
                                       token=HF_TOKEN)

# TWO tokenizers, and the reason cost two silent false negatives last session.
# The custom OmniJudgeTokenizer supplies get_context/parse_response but its
# DECODE is broken under transformers v5: it returns space-separated tokens
# with the byte-BPE marker intact ("F AL GS GE" instead of "FALSE"), so
# parse_response finds none of its anchors and returns all-None -- which reads
# as the judge failing when it is the DECODER failing. AutoTokenizer cannot
# help: the repo declares custom code, so it refuses without
# trust_remote_code and returns the same broken class with it. Build a plain
# fast tokenizer from the repo's own tokenizer.json instead.
_tok_json = hf_hub_download(OM_ID, "tokenizer.json", token=HF_TOKEN)
plain_tok = PreTrainedTokenizerFast(tokenizer_file=_tok_json)
_probe = plain_tok.decode(
    plain_tok("## Equivalence Judgment\nFALSE")["input_ids"],
    skip_special_tokens=True)
assert not pilot.judge.looks_bpe_mangled(_probe), (
    f"the plain tokenizer ALSO mangles the decode: {_probe!r}")
print(f"plain decoder round-trip OK: {_probe!r}")

om_model = AutoModelForCausalLM.from_pretrained(
    OM_ID, torch_dtype=torch.bfloat16, device_map="auto",
    trust_remote_code=True, token=HF_TOKEN).eval()
for m in ("get_context", "parse_response"):
    assert hasattr(om_tok, m), (
        f"tokenizer has no {m}() -- the custom code did not load. Do NOT "
        "hand-build a prompt as a substitute; that would be a different "
        "experiment from the one notebook 26 gated.")

_terminators = [om_tok.eos_token_id]
_eot = om_tok.convert_tokens_to_ids("<|eot_id|>")
if isinstance(_eot, int) and _eot >= 0:
    _terminators.append(_eot)


def omni_judge(item):
    """(verdict, raw text) using the model's OWN prompt builder."""
    ctx = om_tok.get_context(str(run.loc[item, "orig_q"] or ""),
                             str(run.loc[item, "pert_a"] or ""),
                             str(answers[item] or ""))
    enc = om_tok(ctx, return_tensors="pt").to(om_model.device)
    with torch.no_grad():
        out = om_model.generate(
            input_ids=enc["input_ids"], attention_mask=enc["attention_mask"],
            do_sample=False, max_new_tokens=pilot.judge.OMNI_MAX_NEW_TOKENS,
            eos_token_id=_terminators,
            pad_token_id=om_tok.pad_token_id or om_tok.eos_token_id)
    new = out[0][enc["input_ids"].shape[1]:].cpu().tolist()
    text = plain_tok.decode(new, skip_special_tokens=True)
    if pilot.judge.looks_bpe_mangled(text):
        text = pilot.judge.demangle_bpe(om_tok.decode(new, skip_special_tokens=True))
    try:
        parsed = om_tok.parse_response(text)
        judgement = parsed.get("judgement") if isinstance(parsed, dict) else None
    except Exception:
        judgement = None
    verdict = pilot.judge.parse_omni_judgement(judgement)
    # Its own parser returns all-None on this model's real output: the prose
    # decodes fine but the structural markers come back space-corrupted, so
    # the anchors are not there while the verdict plainly is. Falling back is
    # the difference between a finding and a false negative against the judge.
    if verdict == "unclear":
        verdict = pilot.judge.parse_omni_text(text)
    return verdict, text


omni = run_judge("omni", OM_ID, "native", omni_judge)
print(f"\nOmni-Judge: {len(omni)} items")
print(omni["verdict"].value_counts().to_string())
print("\ngate items re-derived (expected 55=incorrect, 273=incorrect):")
for i in pilot.judge.GATE_ITEMS:
    print(f"  item {i}: {omni.loc[i, 'verdict']}")

# DECODE HEALTH. This assert used to call looks_bpe_mangled, which tests for
# ONE signature (the byte-BPE marker) and therefore caught 0 of 300 on the
# 2026-08-12 run while 93% of transcripts were damaged -- it printed "decode
# clean on all 300" and the run completed on garbage. assert_omni_decode_ok
# classifies transposed markers, `#`-shredding and stop-before-verdict too,
# and fails closed. The stored outputs in this notebook PREDATE this fix.
_health = pilot.judge.assert_omni_decode_ok(
    omni["raw_output"].fillna("").tolist())
print(f"decode clean on all {_health['n']}: {_health['counts']}")

del om_model, om_tok, plain_tok
free_gpu()

config.json:   0%|          | 0.00/898 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/55.5k [00:00<?, ?B/s]

tokenization_omnijudge.py:   0%|          | 0.00/6.76k [00:00<?, ?B/s]

[transformers] A new version of the following files was downloaded from https://huggingface.co/KbsdJames/Omni-Judge:
- tokenization_omnijudge.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


tokenizer.json:   0%|          | 0.00/9.09M [00:00<?, ?B/s]

...

Fetching 4 files:   0%|          | 0/4 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/291 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/184 [00:00<?, ?B/s]

omni:   0%|          | 0/300 [00:00<?, ?it/s]


Omni-Judge: 300 items
verdict
unclear      263
incorrect     25
correct       12

gate items re-derived (expected 55=incorrect, 273=incorrect):
  item 55: incorrect
  item 273: incorrect
decode clean on all 300
GPU reclaimed; reserved now 6.17 GB


In [6]:
# --- assemble and write the per-item CSV -----------------------------------
frame = pilot.judge.open_judge_frame(
    run, answers, livemath, omni, v1, v2, human=human,
    answer_types=answer_types)

CSV_NAME = "open_judges_all300_diagnostic_20260812.csv"
CSV_PATH = f"{OUT_DIR}/{CSV_NAME}"
frame.to_csv(CSV_PATH, index=False)
print(f"per-item -> {CSV_PATH}  ({len(frame)} rows)")
print(f"requested columns present and in order: "
      f"{list(frame.columns[:len(pilot.judge.OPEN_JUDGE_COLUMNS)]) == list(pilot.judge.OPEN_JUDGE_COLUMNS)}")
print(f"extras appended after: {list(pilot.judge.OPEN_JUDGE_EXTRA_COLUMNS)}")
print()
print(frame[["livemath_verdict", "omni_verdict"]].apply(
    lambda c: c.value_counts()).fillna(0).astype(int).to_string())

per-item -> /content/drive/MyDrive/uncertainty-math-vlm/audit/open_judges_all300_diagnostic_20260812.csv  (300 rows)
requested columns present and in order: True
extras appended after: ['human_label', 'perception_entropy', 'answer_type', 'livemath_looks_like_solving', 'omni_looks_like_solving', 'omni_invented_reference', 'omni_invented_tokens']

           livemath_verdict  omni_verdict
correct                 261            12
incorrect                39            25
unclear                   0           263


In [7]:
# --- diagnostics + summary. NO ACCURACY IS COMPUTED ANYWHERE BELOW. --------
diag = pilot.judge.open_judge_diagnostics(frame)
rationales = pilot.judge.rationale_sample(frame, n_has_error=8, n_clean=4)

jj = diag["judge_vs_judge"]
print(f"judge vs judge: {jj['agree']}/{jj['n_both_determinate']} = "
      f"{jj['rate']:.1%}  |  chance {jj['chance_rate']:.1%}  |  "
      f"kappa {jj['kappa']:.3f}")
print("  ^ read the kappa, not the rate: two lenient judges agree often")
print("    without agreeing about anything.\n")

for tag in ("livemath", "omni"):
    for rule in ("strict_v1", "strict_v2"):
        d = diag[f"{tag}_vs_{rule}"]
        print(f"{tag:9s} vs {rule}: disagree {d['disagree']}/"
              f"{d['n_determinate']}  (rate {d['rate']:.1%}, "
              f"always-yes baseline {d['always_yes_agreement']:.1%})")
print(f"\nOmni cites a number absent from ALL its inputs on "
      f"{diag['omni_invented_reference_total']}/{diag['n']} items "
      f"(the item-273 signature, as a rate)")

print("\ngate reproduction vs notebooks 24 and 26:")
for tag, probes in diag["gate_reproduction"].items():
    for i, r in probes.items():
        flag = "ok" if r["matches"] else "MISMATCH -- version drift, not a gate failure"
        print(f"  {tag:9s} item {i}: expected {r['expected']:9s} "
              f"observed {r['observed']:9s}  {flag}")

MD_NAME = "open_judges_all300_diagnostic_summary_20260812.md"
MD_PATH = f"{OUT_DIR}/{MD_NAME}"
text = pilot.judge.open_judges_summary_md(MD_PATH, frame, diag, rationales)
print(f"\nsummary -> {MD_PATH}")
print(f"fixed rationale sample (read these by hand): {rationales}")
print("\n" + "=" * 70)
print(text[:3500])
print("\n" + "=" * 70)
print(f"Download {CSV_NAME} and {MD_NAME} from")
print("Drive > uncertainty-math-vlm > audit/ into the repo at")
print("reference/audit/ under the same names.")
print()
print("REMINDER: no number in either file is a model accuracy. Both judges")
print("failed a safety check before this ran; these are scorer diagnostics.")

judge vs judge: 18/37 = 48.6%  |  chance 40.0%  |  kappa 0.144
  ^ read the kappa, not the rate: two lenient judges agree often
    without agreeing about anything.

livemath  vs strict_v1: disagree 146/300  (rate 51.3%, always-yes baseline 47.0%)
livemath  vs strict_v2: disagree 147/300  (rate 51.0%, always-yes baseline 46.0%)
omni      vs strict_v1: disagree 13/37  (rate 64.9%, always-yes baseline 35.1%)
omni      vs strict_v2: disagree 9/37  (rate 75.7%, always-yes baseline 29.7%)

Omni cites a number absent from ALL its inputs on 36/300 items (the item-273 signature, as a rate)

gate reproduction vs notebooks 24 and 26:
  livemath  item 55: expected incorrect observed incorrect  ok
  livemath  item 273: expected correct   observed correct    ok
  omni      item 55: expected incorrect observed incorrect  ok
  omni      item 273: expected incorrect observed incorrect  ok

summary -> /content/drive/MyDrive/uncertainty-math-vlm/audit/open_judges_all300_diagnostic_summary_20260812.md
fi